# FlashNorm fused Triton kernel: prototype v3.2 (two-condition dispatch)

## What's new vs v3.1

v3.1 dispatched on K alone (`K < 1024` -> decode kernel). Problem: forcing SmolLM2 (K=576) M=256/1024/4096 onto the scalar-ops decode kernel produced catastrophic losses (-254% at M=256 up to -4049% at M=4096). Both dimensions matter: tensor cores are required when *either* M or K is large.

v3.2 dispatches on **both** M and K:
- `M < 128 AND K < 1024`: decode kernel (scalar ops are fine; fusion savings win)
- otherwise: GEMM kernel (tensor cores mandatory)

For our three models:
- SmolLM2-135M (K=576): decode at M=1, 16, 64; GEMM at M=256, 1024, 4096
- Llama-3.2-1B (K=2048): GEMM for all M
- Llama-3.1-8B (K=4096): GEMM for all M

Expected: preserves v3.1's SmolLM2 decode wins (+1 to +29%), restores v2's SmolLM2 M=4096 win (+9%), no 1B/8B disasters.

## Pre-flight

1. **Runtime > A100**.
2. Set `INCLUDE_VLLM` based on whether you want the vLLM baseline.
3. Expected wall-clock: 8-12 min.
4. Compute units: 2-3.

## 1. Config

In [ ]:
SHAPES = {
    'SmolLM2-135M':  (576, 960),
    'Llama-3.2-1B':  (2048, 2560),
    'Llama-3.1-8B':  (4096, 6144),
}
M_VALUES = [1, 16, 64, 256, 1024, 4096]
DTYPE = 'float16'
EPS = 1e-6
BENCH_ITERS = 100
WARMUP_ITERS = 20
INCLUDE_VLLM = True

# v3.2: two-condition dispatch. Decode kernel requires BOTH M and K to be small.
# Tensor cores become mandatory when either dimension is large.
M_DECODE_THRESHOLD = 128
K_DECODE_THRESHOLD = 1024

## 2. GPU check

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> A100'
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {name}  |  Memory: {vram:.1f} GB')

## 3. Install + import

In [ ]:
import triton
import triton.language as tl
print(f'triton: {triton.__version__}, torch: {torch.__version__}')
HAS_NATIVE_RMS = hasattr(torch.nn.functional, 'rms_norm')
print(f'torch.nn.functional.rms_norm: {HAS_NATIVE_RMS}')

## 3b. Optional vLLM install

In [ ]:
HAS_VLLM = False
if INCLUDE_VLLM:
    print('Installing vLLM (3-5 min)...')
    import subprocess, sys
    try:
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm'],
                           capture_output=True, text=True, timeout=600)
        if r.returncode != 0:
            print(f'vLLM install failed (rc={r.returncode}). Stderr tail:')
            print('\n'.join(r.stderr.strip().split('\n')[-10:]))
        else:
            try:
                from vllm import _custom_ops as vllm_ops
                HAS_VLLM = True
                print('vLLM ready.')
            except Exception as e:
                print(f'Import failed: {e}')
    except subprocess.TimeoutExpired:
        print('Install timed out.')
print(f'HAS_VLLM = {HAS_VLLM}')

## 4. Reference + baselines

In [ ]:
def ref_flashnorm_qkv(x, W, eps=1e-6):
    x_f32 = x.to(torch.float32)
    rms = torch.sqrt((x_f32 * x_f32).mean(dim=-1, keepdim=True) + eps)
    return (x_f32 / rms).to(x.dtype) @ W

def seq_naive(x, W, eps):
    rms = torch.sqrt((x.float() * x.float()).mean(-1, keepdim=True) + eps).to(x.dtype)
    return (x / rms) @ W

if HAS_NATIVE_RMS:
    def seq_realistic(x, W, eps):
        weight = torch.ones(x.shape[-1], dtype=x.dtype, device=x.device)
        return torch.nn.functional.rms_norm(x, (x.shape[-1],), weight, eps) @ W
else:
    def seq_realistic(x, W, eps):
        variance = x.pow(2).mean(-1, keepdim=True).to(torch.float32)
        inv = torch.rsqrt(variance + eps).to(x.dtype)
        return (x * inv) @ W

if HAS_VLLM:
    def seq_vllm(x, W, eps):
        weight = torch.ones(x.shape[-1], dtype=x.dtype, device=x.device)
        out = torch.empty_like(x)
        vllm_ops.rms_norm(out, x, weight, eps)
        return out @ W
else:
    seq_vllm = None

print('baselines defined')

## 5a. Decode kernel (scalar ops, no tensor cores; for small M AND small K)

In [ ]:
@triton.jit
def _flashnorm_decode_kernel(
    X_ptr, W_ptr, OUT_ptr,
    M, N, K,
    stride_xm, stride_xk,
    stride_wk, stride_wn,
    stride_om, stride_on,
    eps,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)
    n_mask = offs_n < N

    acc = tl.zeros((BLOCK_N,), dtype=tl.float32)
    rms_acc = tl.zeros((), dtype=tl.float32)

    x_ptrs = X_ptr + pid_m * stride_xm + offs_k * stride_xk
    w_ptrs = W_ptr + offs_k[:, None] * stride_wk + offs_n[None, :] * stride_wn

    for k in range(0, tl.cdiv(K, BLOCK_K)):
        k_mask = (k * BLOCK_K + offs_k) < K
        x = tl.load(x_ptrs, mask=k_mask, other=0.0)
        w = tl.load(w_ptrs, mask=k_mask[:, None] & n_mask[None, :], other=0.0)

        x_f32 = x.to(tl.float32)
        w_f32 = w.to(tl.float32)

        rms_acc += tl.sum(x_f32 * x_f32)
        acc += tl.sum(x_f32[:, None] * w_f32, axis=0)

        x_ptrs += BLOCK_K * stride_xk
        w_ptrs += BLOCK_K * stride_wk

    inv_rms = 1.0 / tl.sqrt(rms_acc / K + eps)
    out = acc * inv_rms

    tl.store(OUT_ptr + pid_m * stride_om + offs_n * stride_on,
             out.to(OUT_ptr.dtype.element_ty), mask=n_mask)

print('decode kernel defined')

## 5b. GEMM kernel (tensor cores, autotuned; for large M or large K)

In [ ]:
AUTOTUNE_CONFIGS = [
    triton.Config({'BLOCK_M':  32, 'BLOCK_N': 128, 'BLOCK_K': 32}, num_warps=4, num_stages=3),
    triton.Config({'BLOCK_M':  32, 'BLOCK_N': 256, 'BLOCK_K': 32}, num_warps=4, num_stages=4),
    triton.Config({'BLOCK_M':  64, 'BLOCK_N': 128, 'BLOCK_K': 32}, num_warps=4, num_stages=3),
    triton.Config({'BLOCK_M':  64, 'BLOCK_N': 128, 'BLOCK_K': 64}, num_warps=4, num_stages=3),
    triton.Config({'BLOCK_M':  64, 'BLOCK_N': 256, 'BLOCK_K': 32}, num_warps=4, num_stages=4),
    triton.Config({'BLOCK_M':  64, 'BLOCK_N':  64, 'BLOCK_K': 32}, num_warps=4, num_stages=3),
    triton.Config({'BLOCK_M': 128, 'BLOCK_N': 128, 'BLOCK_K': 32}, num_warps=8, num_stages=3),
    triton.Config({'BLOCK_M': 128, 'BLOCK_N': 128, 'BLOCK_K': 32}, num_warps=4, num_stages=5),
    triton.Config({'BLOCK_M': 128, 'BLOCK_N': 128, 'BLOCK_K': 64}, num_warps=8, num_stages=3),
    triton.Config({'BLOCK_M': 128, 'BLOCK_N': 256, 'BLOCK_K': 32}, num_warps=8, num_stages=3),
    triton.Config({'BLOCK_M': 128, 'BLOCK_N': 256, 'BLOCK_K': 64}, num_warps=8, num_stages=3),
    triton.Config({'BLOCK_M': 256, 'BLOCK_N': 128, 'BLOCK_K': 32}, num_warps=8, num_stages=3),
    triton.Config({'BLOCK_M': 256, 'BLOCK_N': 128, 'BLOCK_K': 64}, num_warps=8, num_stages=3),
]

@triton.autotune(configs=AUTOTUNE_CONFIGS, key=['M', 'N', 'K'])
@triton.jit
def _flashnorm_gemm_kernel(
    X_ptr, W_ptr, OUT_ptr,
    M, N, K,
    stride_xm, stride_xk,
    stride_wk, stride_wn,
    stride_om, stride_on,
    eps,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)
    x_ptrs = X_ptr + offs_m[:, None] * stride_xm + offs_k[None, :] * stride_xk
    w_ptrs = W_ptr + offs_k[:, None] * stride_wk + offs_n[None, :] * stride_wn
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    rms_acc = tl.zeros((BLOCK_M,), dtype=tl.float32)
    m_mask = offs_m < M
    n_mask = offs_n < N
    for k in range(0, tl.cdiv(K, BLOCK_K)):
        k_mask = (k * BLOCK_K + offs_k) < K
        x = tl.load(x_ptrs, mask=m_mask[:, None] & k_mask[None, :], other=0.0)
        w = tl.load(w_ptrs, mask=k_mask[:, None] & n_mask[None, :], other=0.0)
        x_f32 = x.to(tl.float32)
        rms_acc += tl.sum(x_f32 * x_f32, axis=1)
        acc += tl.dot(x, w)
        x_ptrs += BLOCK_K * stride_xk
        w_ptrs += BLOCK_K * stride_wk
    mean_sq = rms_acc / K
    inv_rms = 1.0 / tl.sqrt(mean_sq + eps)
    out = acc * inv_rms[:, None]
    out_ptrs = OUT_ptr + offs_m[:, None] * stride_om + offs_n[None, :] * stride_on
    tl.store(out_ptrs, out.to(OUT_ptr.dtype.element_ty), mask=m_mask[:, None] & n_mask[None, :])

print('gemm kernel defined')

## 5c. Dispatch wrapper (v3.2: two-condition)

Routes to decode kernel only when BOTH M < M_DECODE_THRESHOLD AND K < K_DECODE_THRESHOLD.
Otherwise routes to the autotuned GEMM kernel (tensor cores).

In [ ]:
def flashnorm_qkv(x, W, eps=1e-6):
    M, K = x.shape
    K2, N = W.shape
    assert K == K2
    out = torch.empty((M, N), dtype=x.dtype, device=x.device)

    if M < M_DECODE_THRESHOLD and K < K_DECODE_THRESHOLD:
        # decode path: fixed config, no autotune, no tensor cores.
        # only safe when both M and K are small; large on either axis needs tensor cores.
        BLOCK_N, BLOCK_K = 128, 64
        grid = (M, triton.cdiv(N, BLOCK_N))
        _flashnorm_decode_kernel[grid](
            x, W, out, M, N, K,
            x.stride(0), x.stride(1),
            W.stride(0), W.stride(1),
            out.stride(0), out.stride(1),
            eps,
            BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K, num_warps=4,
        )
    else:
        # prefill / large-K path: autotuned GEMM kernel with tensor cores
        grid = lambda META: (triton.cdiv(M, META['BLOCK_M']), triton.cdiv(N, META['BLOCK_N']))
        _flashnorm_gemm_kernel[grid](
            x, W, out, M, N, K,
            x.stride(0), x.stride(1),
            W.stride(0), W.stride(1),
            out.stride(0), out.stride(1),
            eps,
        )
    return out

print('dispatch wrapper defined (two-condition)')

# PART B -- Correctness (both kernel paths)

In [ ]:
def correctness_check(M, K, N, eps=1e-6, dtype=torch.float16, seed=0):
    torch.manual_seed(seed)
    x = torch.randn(M, K, dtype=dtype, device='cuda')
    W = torch.randn(K, N, dtype=dtype, device='cuda') * 0.02
    ref = ref_flashnorm_qkv(x, W, eps=eps)
    got = flashnorm_qkv(x, W, eps=eps)
    max_abs = (ref.float() - got.float()).abs().max().item()
    cos = torch.nn.functional.cosine_similarity(ref.float().flatten(), got.float().flatten(), dim=0).item()
    path = 'decode' if (M < M_DECODE_THRESHOLD and K < K_DECODE_THRESHOLD) else 'gemm'
    status = 'PASS' if max_abs < 0.05 else 'FAIL'
    print(f'  M={M:>5} K={K:>5} N={N:>5} [{path:>6}]: max_abs={max_abs:.4f} cos={cos:.6f}  {status}')
    return max_abs < 0.05

print('=== correctness ===')
all_pass = True
for name, (K, N) in SHAPES.items():
    for M in [1, 17, 64, 256, 1024, 4096]:
        all_pass &= correctness_check(M, K, N, dtype=torch.float16)
print(f'\nAll passed: {all_pass}')
assert all_pass

# PART C -- Triple-baseline benchmark

In [ ]:
def bench(fn, x, W, eps, iters, warmup):
    for _ in range(warmup):
        fn(x, W, eps)
    torch.cuda.synchronize()
    s = torch.cuda.Event(enable_timing=True); e = torch.cuda.Event(enable_timing=True)
    s.record()
    for _ in range(iters):
        fn(x, W, eps)
    e.record()
    torch.cuda.synchronize()
    return s.elapsed_time(e) / iters

dtype = torch.float16 if DTYPE == 'float16' else torch.bfloat16

hdr = f'{"Model":<15} {"M":>6} {"Path":>7} {"Naive":>9} {"Realistic":>11}'
if HAS_VLLM: hdr += f' {"vLLM":>9}'
hdr += f' {"Triton":>9} {"vs Naive":>10} {"vs Realistic":>14}'
if HAS_VLLM: hdr += f' {"vs vLLM":>10}'
print(hdr)
print('-' * len(hdr))

results = {}
for name, (K, N) in SHAPES.items():
    for M in M_VALUES:
        torch.manual_seed(0)
        x = torch.randn(M, K, dtype=dtype, device='cuda')
        W = torch.randn(K, N, dtype=dtype, device='cuda') * 0.02

        path = 'decode' if (M < M_DECODE_THRESHOLD and K < K_DECODE_THRESHOLD) else 'gemm'
        t_n = bench(seq_naive, x, W, EPS, BENCH_ITERS, WARMUP_ITERS)
        t_r = bench(seq_realistic, x, W, EPS, BENCH_ITERS, WARMUP_ITERS)
        t_v = bench(seq_vllm, x, W, EPS, BENCH_ITERS, WARMUP_ITERS) if HAS_VLLM else float('nan')
        t_t = bench(flashnorm_qkv, x, W, EPS, BENCH_ITERS, WARMUP_ITERS)

        sp_n = (t_n - t_t) / t_n * 100
        sp_r = (t_r - t_t) / t_r * 100
        sp_v = (t_v - t_t) / t_v * 100 if HAS_VLLM else float('nan')

        results[(name, M)] = (path, t_n, t_r, t_v, t_t, sp_n, sp_r, sp_v)

        row = f'{name:<15} {M:>6d} {path:>7} {t_n:>9.4f} {t_r:>11.4f}'
        if HAS_VLLM: row += f' {t_v:>9.4f}'
        row += f' {t_t:>9.4f} {sp_n:>+9.1f}% {sp_r:>+13.1f}%'
        if HAS_VLLM: row += f' {sp_v:>+9.1f}%'
        print(row)
        del x, W
        torch.cuda.empty_cache()

# PART D -- Summary

In [ ]:
import math
def fmt(v, p=4):
    return 'nan' if (isinstance(v, float) and math.isnan(v)) else f'{v:.{p}f}'
def fmt_d(v):
    return 'nan' if (isinstance(v, float) and math.isnan(v)) else f'{v:+.1f}%'

print('=' * 115)
print('SUMMARY v3.2: two-condition dispatch (M < 128 AND K < 1024 -> decode; else GEMM)')
print('=' * 115)
hdr = f'{"Model":<15} {"M":>6} {"Path":>7} {"Naive":>9} {"Realistic":>11}'
if HAS_VLLM: hdr += f' {"vLLM":>9}'
hdr += f' {"Triton":>9} {"vs Naive":>10} {"vs Realistic":>14}'
if HAS_VLLM: hdr += f' {"vs vLLM":>10}'
print(hdr)
print('-' * len(hdr))
for (name, M), (path, tn, tr, tv, tt, spn, spr, spv) in results.items():
    line = f'{name:<15} {M:>6d} {path:>7} {fmt(tn):>9} {fmt(tr):>11}'
    if HAS_VLLM: line += f' {fmt(tv):>9}'
    line += f' {fmt(tt):>9} {fmt_d(spn):>10} {fmt_d(spr):>14}'
    if HAS_VLLM: line += f' {fmt_d(spv):>10}'
    print(line)

print()
print('Expected shape of results:')
print('  SmolLM2-135M (K=576):    decode at M=1/16/64; GEMM at M=256/1024/4096')
print('  Llama-3.2-1B (K=2048):   GEMM at all M (K >= 1024)')
print('  Llama-3.1-8B (K=4096):   GEMM at all M (K >= 1024)')

## Optional: auto-disconnect

In [ ]:
import time
print('\n[auto-disconnect] benchmark done. Disconnecting in 90 seconds.')
print('[auto-disconnect] press the stop button to cancel.')
for i in range(9, 0, -1):
    time.sleep(10)
    print(f'[auto-disconnect] {i * 10}s remaining...')
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print(f'[auto-disconnect] failed: {e}')
    print('[auto-disconnect] manually: Runtime > Disconnect and delete runtime')